# OptiFrame — modèle de segmentation du verre (Palier 2)

Ce notebook entraîne le petit U-Net qui repère le verre dans la **zone redressée**
de la feuille de capture, puis l'exporte au format TensorFlow.js pour qu'il tourne
dans le navigateur du téléphone, sans serveur.

## Ce que le modèle fait, et ce qu'il ne fait pas

Il ne mesure pas. Il entre à 256 × 256 pixels pour une zone de 156 × 176 mm, donc
un de ses pixels vaut 0,6 mm : il ne pourra jamais placer un bord au dixième de
millimètre. Son travail est de dire **où est le verre**, de façon robuste, y
compris quand un reflet a effacé une partie du bord.

C'est le détecteur de bord classique (`segmentByRimRelief` + tracé radial par
programmation dynamique, voir `src/lib/pipeline/`) qui place ensuite le contour
au sous-pixel, en partant du centre que le modèle a fourni. Le modèle oriente,
le bord mesure. Si le bord est inexploitable, l'app retombe sur le contour du
modèle et l'affiche comme tel.

## Jeu de données

| Source | Volume | Licence |
|---|---|---|
| Rendus synthétiques (`scripts/synthetic.py`, code du dépôt) | ~3000 | ce dépôt |
| Photos réelles de nos propres verres, masques tracés à la main (`scripts/label_lens.py`) | à compléter | ce dépôt, aucune donnée personnelle |

Aucun jeu de données externe, aucun modèle pré-entraîné : tout est produit ici,
donc il n'y a aucune licence tierce à citer. Les rendus synthétiques modélisent
ce qui rend un verre visible — le biseau du bord qui assombrit et renvoie une
caustique, la perte de contraste de Fresnel d'environ 8 % à la traversée, les
reflets spéculaires, l'ombre portée — et **pas** une réfraction spectaculaire :
posé sur le papier, un verre de 4 dioptries ne décale ce qu'il y a dessous que
d'une vingtaine de micromètres.


In [ ]:
!pip -q install tensorflow tensorflowjs
import os, glob, json, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print('TensorFlow', tf.__version__)


## 1. Charger les paires (image, masque)

Produites en local par :

```bash
node scripts/headless/build.mjs
python3 scripts/make_dataset.py --synthetic 3000 --out data/train
python3 scripts/label_lens.py --photos pictures --out data/real   # photos réelles
```

puis zippées et déposées ici (ou montées depuis Drive).


In [ ]:
DATA_DIRS = ['data/train', 'data/real']   # adapter si besoin
INPUT_SIZE = 256                           # doit valoir INPUT_SIZE dans src/lib/model/segmenter.ts

def load_pairs(dirs):
    images, masks, origins = [], [], []
    for d in dirs:
        for image_path in sorted(glob.glob(os.path.join(d, 'images', '*.png'))):
            mask_path = image_path.replace('/images/', '/masks/')
            if not os.path.exists(mask_path):
                continue
            images.append(tf.io.decode_png(tf.io.read_file(image_path), channels=1).numpy())
            masks.append(tf.io.decode_png(tf.io.read_file(mask_path), channels=1).numpy())
            origins.append(os.path.basename(d))
    return np.array(images, np.float32) / 255.0, (np.array(masks, np.float32) / 255.0 > 0.5).astype(np.float32), np.array(origins)

X, Y, ORIGIN = load_pairs(DATA_DIRS)
print(X.shape, Y.shape, 'couverture moyenne du masque : %.3f' % Y.mean())


## 2. Séparation train / validation

Les photos réelles sont rares et sont ce qui compte : on en garde une part
suffisante en validation pour que le chiffre rapporté dans le README veuille
dire quelque chose. La validation ne contient **que** du réel quand il y en a —
un modèle qui excelle sur des rendus et échoue sur du papier n'aurait rien appris
d'utile.


In [ ]:
rng = np.random.default_rng(0)
real = np.flatnonzero(ORIGIN == 'real')
synth = np.flatnonzero(ORIGIN != 'real')

if len(real) >= 20:
    rng.shuffle(real)
    cut = max(8, len(real) // 4)
    val_idx, train_idx = real[:cut], np.concatenate([real[cut:], synth])
else:
    # Pas encore assez de photos réelles : on valide sur du synthétique tenu à
    # l'écart, et on le dit explicitement dans le README.
    rng.shuffle(synth)
    cut = max(1, len(synth) // 10)
    val_idx, train_idx = synth[:cut], np.concatenate([synth[cut:], real])

print(f'{len(train_idx)} entraînement, {len(val_idx)} validation '
      f'({(ORIGIN[val_idx] == "real").sum()} photos réelles en validation)')


## 3. Augmentation

Les variations géométriques et photométriques que le rendu ne couvre pas
déjà : le verre peut être posé n'importe où dans la zone et dans les deux sens,
et l'exposition d'un téléphone varie d'une prise à l'autre.


In [ ]:
def augment(image, mask):
    if tf.random.uniform(()) < 0.5:
        image, mask = tf.image.flip_left_right(image), tf.image.flip_left_right(mask)
    if tf.random.uniform(()) < 0.5:
        image, mask = tf.image.flip_up_down(image), tf.image.flip_up_down(mask)
    k = tf.random.uniform((), 0, 4, tf.int32)
    image, mask = tf.image.rot90(image, k), tf.image.rot90(mask, k)
    image = image * tf.random.uniform((), 0.75, 1.25) + tf.random.uniform((), -0.12, 0.12)
    image = image + tf.random.normal(tf.shape(image), 0.0, tf.random.uniform((), 0.0, 0.03))
    return tf.clip_by_value(image, 0.0, 1.0), mask

BATCH = 16
train_ds = (tf.data.Dataset.from_tensor_slices((X[train_idx], Y[train_idx]))
            .shuffle(2048).map(augment, tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X[val_idx], Y[val_idx])).batch(BATCH)


## 4. Le modèle

Un U-Net volontairement minuscule : environ 150 000 paramètres, moins d'un
mégaoctet une fois converti. Il doit tourner dans un Web Worker sur le téléphone
du jury, sur le backend WebAssembly de TensorFlow.js, et laisser de la marge
dans les trente secondes que le défi accorde pour une paire de verres.

Un modèle plus gros segmenterait un peu mieux et ne servirait à rien : c'est le
tracé du bord, en aval, qui fixe la précision finale.


In [ ]:
def conv_block(x, filters):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x

def build_unet(size=INPUT_SIZE, base=12):
    inputs = keras.Input((size, size, 1))
    skips = []
    x = inputs
    for depth in range(4):
        x = conv_block(x, base * 2 ** depth)
        skips.append(x)
        x = layers.MaxPool2D()(x)
    x = conv_block(x, base * 16)
    for depth in reversed(range(4)):
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, skips[depth]])
        x = conv_block(x, base * 2 ** depth)
    outputs = layers.Conv2D(1, 1, activation='sigmoid', name='mask')(x)
    return keras.Model(inputs, outputs, name='optiframe_unet')

def dice_bce(y_true, y_pred):
    # Dice en plus de l'entropie croisée : le verre couvre ~5 % de l'image, et
    # une BCE seule est parfaitement satisfaite par un masque entièrement vide.
    smooth = 1.0
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true, [1, 2, 3]) + tf.reduce_sum(y_pred, [1, 2, 3])
    dice = 1 - (2 * intersection + smooth) / (union + smooth)
    return tf.reduce_mean(dice) + keras.losses.binary_crossentropy(y_true, y_pred)

def iou(y_true, y_pred):
    p = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * p, [1, 2, 3])
    union = tf.reduce_sum(tf.cast((y_true + p) > 0, tf.float32), [1, 2, 3])
    return tf.reduce_mean((inter + 1e-6) / (union + 1e-6))

model = build_unet()
model.compile(optimizer=keras.optimizers.Adam(2e-3), loss=dice_bce, metrics=[iou])
print(f'{model.count_params():,} paramètres')


In [ ]:
history = model.fit(
    train_ds, validation_data=val_ds, epochs=40,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau('val_iou', mode='max', factor=0.5, patience=4),
        keras.callbacks.EarlyStopping('val_iou', mode='max', patience=10, restore_best_weights=True),
    ],
)
print('meilleur IoU validation : %.4f' % max(history.history['val_iou']))


## 5. Mesurer la performance honnêtement

L'IoU n'est pas la métrique du barème : le jury compare A et B au pied à
coulisse. On convertit donc l'IoU en ce qui compte — l'erreur sur la boîte
englobante, en millimètres réels — en utilisant l'échelle de la zone redressée.

Le chiffre obtenu ici est celui du **modèle seul**. Dans l'app, le tracé de bord
affine ensuite ce contour ; la mesure de bout en bout se vérifie avec
`python3 scripts/eval_mm.py`, qui fait tourner le vrai pipeline.


In [ ]:
ZONE_WIDTH_MM, ZONE_HEIGHT_MM = 156.0, 176.0   # zone de pose moins la marge, cf. runPipeline.ts
mm_per_px_x, mm_per_px_y = ZONE_WIDTH_MM / INPUT_SIZE, ZONE_HEIGHT_MM / INPUT_SIZE

def bbox_mm(mask):
    ys, xs = np.nonzero(mask[:, :, 0] > 0.5)
    if len(xs) == 0:
        return None
    return ((xs.max() - xs.min()) * mm_per_px_x, (ys.max() - ys.min()) * mm_per_px_y)

pred = model.predict(X[val_idx], batch_size=BATCH, verbose=0)
errors_a, errors_b, missed = [], [], 0
for p, t in zip(pred, Y[val_idx]):
    bp, bt = bbox_mm(p), bbox_mm(t)
    if bp is None or bt is None:
        missed += 1
        continue
    errors_a.append(bp[0] - bt[0]); errors_b.append(bp[1] - bt[1])

errors = np.abs(errors_a + errors_b)
print(f'IoU validation          : {iou(Y[val_idx], pred).numpy():.4f}')
print(f'erreur A/B du modèle    : {errors.mean():.2f} mm en moyenne, p95 {np.percentile(errors, 95):.2f} mm')
print(f'non détecté             : {missed}/{len(val_idx)}')
print('Rappel : ce chiffre est celui du modèle seul (0,6 mm par pixel).')
print("Le chiffre de bout en bout vient de scripts/eval_mm.py et doit figurer dans le README.")


## 6. Export vers TensorFlow.js

L'app charge un **graph model** (`tf.loadGraphModel`) depuis `public/model/`.


In [ ]:
import tensorflowjs as tfjs, shutil, os

model.export('saved_model')
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model \
    --signature_name=serving_default --saved_model_tags=serve \
    saved_model tfjs_model

total = sum(os.path.getsize(os.path.join('tfjs_model', f)) for f in os.listdir('tfjs_model'))
print(f'modèle exporté : {total / 1e6:.2f} Mo')
shutil.make_archive('optiframe_model', 'zip', 'tfjs_model')
print('Téléchargez optiframe_model.zip, décompressez-le dans public/model/ du dépôt,')
print('puis vérifiez de bout en bout : npm run build && python3 scripts/eval_mm.py --synthetic 40')


## 7. À reporter dans le README

- nombre de rendus synthétiques et de photos réelles utilisés ;
- IoU de validation et erreur A/B du modèle seul (section 5) ;
- erreur de bout en bout mesurée par `scripts/eval_mm.py` sur vos propres
  verres mesurés au pied à coulisse ;
- taille du modèle et temps d'inférence observé sur un vrai téléphone.

Si le modèle fait **moins bien** que le chemin classique, c'est à dire et à
garder : le barème récompense les limites reconnues honnêtement, et l'app
fonctionne sans lui.
